# MedVS-AI — Benign/malignant classifier (Phase 1, option 2)

Trains an EfficientNet-B0 to call a dermoscopy lesion **benign vs malignant** (ISIC 2018 Task 3 /
HAM10000) and exports `classifier.onnx`. Drop it into `ml/models/classifier.onnx` and the web app's
model stops abstaining — the diagnosis half becomes a real head-to-head.

**Set a T4 GPU** (`Runtime > Change runtime type`), then `Runtime > Run all`.
Malignant = {MEL, BCC, AKIEC}; benign = {NV, BKL, DF, VASC}.
> NOT FOR CLINICAL USE.

## 1. GPU check

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if not torch.cuda.is_available(): print("Set Runtime > Change runtime type > T4 GPU")


## 2. Install extras

In [ ]:
%pip -q install onnxscript onnxruntime scikit-learn
print("ok")


## 3. Config

In [ ]:
import torch
IMG_SIZE, BATCH, EPOCHS, LR, SEED = 256, 32, 8, 3e-4, 42
MALIGNANT = {"MEL", "BCC", "AKIEC"}
MEAN = (0.485, 0.456, 0.406); STD = (0.229, 0.224, 0.225)
T3_IMAGES = "https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task3_Training_Input.zip"
T3_GT     = "https://isic-challenge-data.s3.amazonaws.com/2018/ISIC2018_Task3_Training_GroundTruth.zip"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device", DEVICE)


## 4. Download ISIC 2018 Task 3 (HAM10000)

In [ ]:
import os, zipfile, urllib.request
os.makedirs("t3", exist_ok=True)
def fetch(url, zname, marker):
    if os.path.isdir(marker) and os.listdir(marker): print("have", marker); return
    if not os.path.exists(zname): print("downloading", url); urllib.request.urlretrieve(url, zname)
    with zipfile.ZipFile(zname) as z: z.extractall("t3")
    print("extracted", zname)
fetch(T3_GT, "t3_gt.zip", "t3/ISIC2018_Task3_Training_GroundTruth")
fetch(T3_IMAGES, "t3_img.zip", "t3/ISIC2018_Task3_Training_Input")
IMG_DIR = "t3/ISIC2018_Task3_Training_Input"
GT_CSV  = "t3/ISIC2018_Task3_Training_GroundTruth/ISIC2018_Task3_Training_GroundTruth.csv"
print("images:", len(os.listdir(IMG_DIR)))


## 5. Labels → benign/malignant + split
*Optional but recommended:* upload **`HAM10000_metadata.csv`** via Colab's file panel for a **lesion-grouped** split (no leakage — the same lesion never lands in both train and val). Without it, the split is image-level and mildly optimistic. (Get it from Kaggle `kmader/skin-cancer-mnist-ham10000` or Harvard Dataverse DOI `10.7910/DVN/DBW86T`.)

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv(GT_CSV)
CLASSES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
df["dx_name"] = [CLASSES[i] for i in df[CLASSES].values.argmax(1)]
df["label"] = df["dx_name"].isin(MALIGNANT).astype(int)
df["path"] = df["image"].map(lambda x: os.path.join(IMG_DIR, x + ".jpg"))
df = df[df["path"].map(os.path.exists)].reset_index(drop=True)
print("total", len(df), "| malignant", int(df.label.sum()), "| benign", int((1 - df.label).sum()))

# Split: LESION-GROUPED if HAM10000_metadata.csv is present (no leakage), else stratified image-level.
META = "HAM10000_metadata.csv"   # upload via Colab's file panel (Kaggle: kmader/skin-cancer-mnist-ham10000)
if os.path.exists(META):
    from sklearn.model_selection import GroupShuffleSplit
    meta = pd.read_csv(META)[["image_id", "lesion_id"]]
    df = df.merge(meta, left_on="image", right_on="image_id", how="left")
    df["lesion_id"] = df["lesion_id"].fillna(df["image"])
    tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
                          .split(df, df["label"], groups=df["lesion_id"]))
    tr, va = df.iloc[tr_idx].reset_index(drop=True), df.iloc[va_idx].reset_index(drop=True)
    assert set(tr["lesion_id"]).isdisjoint(set(va["lesion_id"]))
    print(f"LESION-GROUPED split (no leakage): train {len(tr)} / val {len(va)} | {df['lesion_id'].nunique()} lesions")
else:
    from sklearn.model_selection import train_test_split
    tr, va = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])
    print(f"IMAGE-LEVEL split: train {len(tr)} / val {len(va)}")
    print("WARNING: no HAM10000_metadata.csv -> lesions can repeat across train/val (mild optimism).")
    print("         Upload HAM10000_metadata.csv for a leakage-free lesion-grouped split.")


## 6. Train EfficientNet-B0 (single logit = P(malignant))

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import roc_auc_score
from PIL import Image
mean = np.array(MEAN, np.float32); std = np.array(STD, np.float32)

class DS(Dataset):
    def __init__(self, frame, aug=False): self.f = frame.reset_index(drop=True); self.aug = aug
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]
        im = Image.open(r["path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        x = np.asarray(im, np.float32) / 255.0
        if self.aug and np.random.rand() < 0.5: x = x[:, ::-1, :].copy()
        x = (x - mean) / std
        return torch.from_numpy(x.transpose(2, 0, 1)), torch.tensor([r["label"]], dtype=torch.float32)

tr_dl = DataLoader(DS(tr, aug=True), batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
va_dl = DataLoader(DS(va), batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
model = model.to(DEVICE)
pos_weight = torch.tensor([(tr.label == 0).sum() / max((tr.label == 1).sum(), 1)], dtype=torch.float32, device=DEVICE)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.Adam(model.parameters(), lr=LR)

def evaluate():
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in va_dl:
            p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy().ravel()
            ps += p.tolist(); ys += y.numpy().ravel().tolist()
    return roc_auc_score(ys, ps), np.array(ys), np.array(ps)

best = -1.0
for e in range(1, EPOCHS + 1):
    model.train(); losses = []
    for x, y in tr_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step(); losses.append(loss.item())
    auc, _, _ = evaluate()
    tag = ""
    if auc > best: best = auc; torch.save(model.state_dict(), "classifier.pt"); tag = " *saved"
    print(f"epoch {e}/{EPOCHS}  loss={np.mean(losses):.4f}  val_AUROC={auc:.4f}{tag}")
print("best val AUROC:", round(best, 4))


## 7. Evaluate (sensitivity matters most for malignant)

In [ ]:
model.load_state_dict(torch.load("classifier.pt"))
auc, ys, ps = evaluate()
pred = (ps >= 0.5).astype(int)
tp = int(((pred == 1) & (ys == 1)).sum()); fn = int(((pred == 0) & (ys == 1)).sum())
tn = int(((pred == 0) & (ys == 0)).sum()); fp = int(((pred == 1) & (ys == 0)).sum())
print(f"AUROC={auc:.3f}  malignant sensitivity={tp/max(tp+fn,1):.3f}  specificity={tn/max(tn+fp,1):.3f}")
print(f"confusion: TP={tp} FN={fn} TN={tn} FP={fp}")


## 7b. Calibration (temperature scaling)
Fits a single temperature on the validation set and reports **ECE** (expected calibration error) before vs. after — so the model's confidence is honest, not just its accuracy.

In [ ]:
# Temperature scaling for honest confidence (does NOT change the 0.5 benign/malignant call).
import torch
def ece(probs, labels, bins=10):
    probs, labels = np.asarray(probs), np.asarray(labels); e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for i in range(bins):
        m = (probs > edges[i]) & (probs <= edges[i + 1])
        if m.sum(): e += m.mean() * abs(labels[m].mean() - probs[m].mean())
    return float(e)

model.eval(); logits, labs = [], []
with torch.no_grad():
    for x, y in va_dl:
        logits += model(x.to(DEVICE)).cpu().numpy().ravel().tolist()
        labs += y.numpy().ravel().tolist()
logits, labs = np.array(logits), np.array(labs)

T = torch.nn.Parameter(torch.ones(1))
lt, ly = torch.tensor(logits, dtype=torch.float32), torch.tensor(labs, dtype=torch.float32)
optT = torch.optim.LBFGS([T], lr=0.1, max_iter=60); bce_t = torch.nn.BCEWithLogitsLoss()
def _closure():
    optT.zero_grad(); loss = bce_t(lt / T, ly); loss.backward(); return loss
optT.step(_closure); T_val = float(T.detach())

p_before = 1 / (1 + np.exp(-logits)); p_after = 1 / (1 + np.exp(-logits / T_val))
print(f"temperature T = {T_val:.3f}")
print(f"ECE  before = {ece(p_before, labs):.4f}   ->   after = {ece(p_after, labs):.4f}")
print("(T rescales probabilities for calibration; the 0.5 benign/malignant decision is unchanged.)")


## Done
Put `classifier.onnx` at `ml/models/classifier.onnx`, then reseed so the model's diagnosis is stored:
```
cd web/backend
python seed_cases.py --bundle ../data/bundle     # or --synthetic 20
python app.py
```
The reveal now shows the model's benign/malignant call, whether it was right, and whether you agreed.

**Caveats:** benign/malignant is a coarse grouping of the 7 ISIC classes; use a **lesion-grouped split**
(upload `HAM10000_metadata.csv`) to avoid the mild optimism of an image-level split; the AUROC is
honest but the calibration (ECE) tells you whether the *confidence* is trustworthy. Research/education,
not a diagnostic device.

In [ ]:
model.eval().cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(model, dummy, "classifier.onnx",
                  input_names=["input"], output_names=["logit"],
                  dynamic_axes={"input": {0: "batch"}}, opset_version=17, dynamo=False)
import onnxruntime as ort
sess = ort.InferenceSession("classifier.onnx", providers=["CPUExecutionProvider"])
o = sess.run(None, {"input": dummy.numpy()})[0]
print("onnx output shape", o.shape, "(expect (1, 1)) -> single logit OK")
from google.colab import files
files.download("classifier.onnx")


## Done
Put `classifier.onnx` at `ml/models/classifier.onnx`, then reseed so the model's diagnosis is stored:
```
cd web/backend
python seed_cases.py --bundle ../data/bundle     # or --synthetic 20
python app.py
```
The reveal now shows the model's benign/malignant call, whether it was right, and whether you agreed.

**Caveats:** benign/malignant is a coarse grouping of the 7 ISIC classes; the split is image-level
(HAM lesions can repeat across images, so expect mild optimism); this is research/education, not a
diagnostic device.